# Лабораторная работа 4

Цель работы — реализовать собственную функцию `bootstrap`, оценить параметры распределений из ЛР №1 и сравнить результаты со встроенной функцией `scipy.stats.bootstrap`.

Во всех вычислениях ниже используются те же выборки, что и в ЛР №1:

- `U(2,7)` при `n=100` и `n=1000`;
- `Bernoulli(0.27)` при `n=100` и `n=1000`;
- `Bin(12, 0.35)` при `n=100` и `n=1000`;
- `N(4.5, 1.8^2)` при `n=100` и `n=1000`.

Для сравнения со SciPy ниже используется именно percentile-bootstrap, потому что он соответствует алгоритму задания. Важно: в `scipy.stats.bootstrap` по умолчанию стоит метод `BCa`, поэтому мы явно задаём `method="percentile"`.

## Метод bootstrap

Пусть дана выборка $X_1, X_2, \dots, X_n$ из распределения $F$ и требуется построить доверительный интервал для параметра $\theta$.

1. **Вычисляем точечную оценку $\widehat{\theta}_n$.**

   Если статистика $\widehat{\theta}_n = T(X_1, \dots, X_n)$ является состоятельной, то
   $$
   \widehat{\theta}_n \xrightarrow{P} \theta, \qquad n \to \infty.
   $$
   Это означает, что при росте объёма выборки оценка концентрируется около истинного параметра.

2. **Генерируем bootstrap-выборки с возвращением.**

   При таком пересэмплировании мы на самом деле моделируем выборки не из неизвестного распределения $F$, а из эмпирического распределения
   $$
   \widehat{F}_n(x) = \frac{1}{n} \sum_{i=1}^{n} \mathbf{1}\{X_i \le x\}.
   $$
   По теореме Гливенко–Кантелли
   $$
   \sup_x |\widehat{F}_n(x) - F(x)| \xrightarrow{a.s.} 0,
   $$
   поэтому эмпирическое распределение асимптотически хорошо приближает истинное.

3. **На каждой bootstrap-выборке заново считаем оценку.**

   Для $i$-й bootstrap-выборки получаем
   $$
   \theta_i^* = T(X_1^{*(i)}, X_2^{*(i)}, \dots, X_n^{*(i)}).
   $$
   Условное распределение $\theta_i^*$ при фиксированной исходной выборке приближает распределение самой статистики $\widehat{\theta}_n$. Идея bootstrap состоит в том, что разброс оценок на повторных пересэмплированиях воспроизводит разброс исходной оценки.

4. **Берём квантили bootstrap-распределения.**

   Если уровень доверия равен $\gamma$, а $\alpha = 1 - \gamma$, то percentile-интервал строится по эмпирическим квантилям bootstrap-оценок:
   $$
   \left( \theta^*_{\left(\lfloor \frac{\alpha}{2} k \rfloor\right)},\ \theta^*_{\left(\lceil \left(1 - \frac{\alpha}{2}\right) k \rceil\right)} \right).
   $$
   Эмпирические квантили bootstrap-распределения сходятся к квантилям распределения статистики, поэтому покрытие такого интервала асимптотически стремится к $\gamma$.

## Состоятельные оценки параметров

Обозначим
$$
\overline{X} = \frac{1}{n}\sum_{i=1}^{n} X_i,
\qquad
S_n^2 = \frac{1}{n}\sum_{i=1}^{n}(X_i - \overline{X})^2.
$$
По закону больших чисел
$$
\overline{X} \xrightarrow{P} \mathbb{E}X,
\qquad
S_n^2 \xrightarrow{P} \mathrm{Var}(X).
$$
Дальше используется теорема о непрерывном отображении.

### 1. Равномерное распределение $U(a,b)$

Для равномерного распределения
$$
\mathbb{E}X = \frac{a+b}{2},
\qquad
\mathrm{Var}(X) = \frac{(b-a)^2}{12}.
$$
Отсюда метод моментов даёт оценки
$$
\widehat{a} = \overline{X} - \sqrt{3S_n^2},
\qquad
\widehat{b} = \overline{X} + \sqrt{3S_n^2}.
$$
Так как обе формулы являются непрерывными функциями от $(\overline{X}, S_n^2)$, оценки $\widehat{a}$ и $\widehat{b}$ состоятельны.

### 2. Распределение Бернулли $Bernoulli(p)$

Здесь
$$
\mathbb{E}X = p,
$$
поэтому естественная оценка
$$
\widehat{p} = \overline{X}
$$
состоятельна напрямую по закону больших чисел.

### 3. Биномиальное распределение $Bin(m,p)$

Для биномиального распределения
$$
\mathbb{E}X = mp,
\qquad
\mathrm{Var}(X) = mp(1-p).
$$
Решая систему уравнений метода моментов, получаем
$$
\widehat{m} = \frac{\overline{X}^{2}}{\overline{X} - S_n^2},
\qquad
\widehat{p} = \frac{\overline{X}}{\widehat{m}}.
$$
Поскольку $(\overline{X}, S_n^2) \xrightarrow{P} (mp, mp(1-p))$, имеем $\widehat{m} \xrightarrow{P} m$ и $\widehat{p} \xrightarrow{P} p$. В коде $\widehat{m}$ оставляется вещественной: так bootstrap-распределение получается менее дискретным и сравнение со SciPy становится стабильнее. При желании эту оценку можно округлить уже после расчётов.

### 4. Нормальное распределение $N(\mu, \sigma^2)$

Для нормального распределения используем оценки
$$
\widehat{\mu} = \overline{X},
\qquad
\widehat{\sigma} = \sqrt{S_n^2}.
$$
Состоятельность $\widehat{\mu}$ следует из закона больших чисел, а состоятельность $\widehat{\sigma}$ — из сходимости $S_n^2$ к $\sigma^2$ и непрерывности функции квадратного корня.

In [1]:
import math
import numpy as np
import pandas as pd
import scipy.stats as st
from IPython.display import Markdown, display

RANDOM_STATE = 16
BOOTSTRAP_RANDOM_STATE = 2026
CONFIDENCE_LEVEL = 0.95
N_RESAMPLES = 10_000

pd.set_option("display.max_colwidth", None)


def build_samples(random_state=RANDOM_STATE):
    return {
        "uniform_100": st.uniform.rvs(loc=2, scale=5, size=100, random_state=random_state),
        "uniform_1000": st.uniform.rvs(loc=2, scale=5, size=1000, random_state=random_state),
        "bernoulli_100": st.bernoulli.rvs(p=0.27, size=100, random_state=random_state),
        "bernoulli_1000": st.bernoulli.rvs(p=0.27, size=1000, random_state=random_state),
        "binom_100": st.binom.rvs(n=12, p=0.35, size=100, random_state=random_state),
        "binom_1000": st.binom.rvs(n=12, p=0.35, size=1000, random_state=random_state),
        "norm_100": st.norm.rvs(loc=4.5, scale=1.8, size=100, random_state=random_state),
        "norm_1000": st.norm.rvs(loc=4.5, scale=1.8, size=1000, random_state=random_state),
    }


samples = build_samples()


def sample_mean(sample):
    sample = np.asarray(sample, dtype=float)
    return float(np.mean(sample))


def sample_var(sample):
    sample = np.asarray(sample, dtype=float)
    centered = sample - np.mean(sample)
    return float(np.mean(centered ** 2))


def estimate_uniform_params(sample):
    mu = sample_mean(sample)
    var = sample_var(sample)
    half_width = math.sqrt(max(3.0 * var, 0.0))
    return mu - half_width, mu + half_width


def estimate_uniform_a(sample):
    return estimate_uniform_params(sample)[0]


def estimate_uniform_b(sample):
    return estimate_uniform_params(sample)[1]


def estimate_bernoulli_p(sample):
    return sample_mean(sample)


def estimate_binom_params(sample):
    x = np.asarray(sample, dtype=float)
    mu = sample_mean(x)
    var = sample_var(x)
    eps = 1e-12

    if mu <= eps:
        return 1.0, 0.0

    denominator = mu - var
    if denominator <= eps:
        m_hat = max(float(np.max(x)), 1.0)
    else:
        m_hat = max(mu * mu / denominator, float(np.max(x)), 1.0)

    p_hat = min(max(mu / m_hat, eps), 1 - eps)
    return float(m_hat), float(p_hat)


def estimate_binom_m(sample):
    return estimate_binom_params(sample)[0]


def estimate_binom_p(sample):
    return estimate_binom_params(sample)[1]


def estimate_normal_mu(sample):
    return sample_mean(sample)


def estimate_normal_sigma(sample):
    return math.sqrt(max(sample_var(sample), 0.0))


def bootstrap_percentile(sample, estimator, confidence_level=0.95, n_resamples=9999, random_state=0):
    sample = np.asarray(sample)
    n = sample.size
    rng = np.random.default_rng(random_state)

    theta_hat = float(estimator(sample))
    bootstrap_estimates = np.empty(n_resamples, dtype=float)

    for i in range(n_resamples):
        bootstrap_sample = rng.choice(sample, size=n, replace=True)
        bootstrap_estimates[i] = float(estimator(bootstrap_sample))

    bootstrap_estimates.sort()
    alpha = 1.0 - confidence_level
    lower_rank = max(1, int(np.floor(alpha / 2.0 * n_resamples)))
    upper_rank = min(n_resamples, int(np.ceil((1.0 - alpha / 2.0) * n_resamples)))

    return {
        "estimate": theta_hat,
        "confidence_interval": (
            float(bootstrap_estimates[lower_rank - 1]),
            float(bootstrap_estimates[upper_rank - 1]),
        ),
        "bootstrap_distribution": bootstrap_estimates,
    }


def scipy_percentile_bootstrap(sample, estimator, confidence_level=0.95, n_resamples=9999, random_state=0):
    result = st.bootstrap(
        (np.asarray(sample),),
        estimator,
        n_resamples=n_resamples,
        confidence_level=confidence_level,
        method="percentile",
        vectorized=False,
        random_state=random_state,
    )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


EXPERIMENTS = [
    {
        "sample_key": "uniform_100",
        "label": "U(2,7), n=100",
        "true_parameters": {"a": 2.0, "b": 7.0},
        "estimators": {"a": estimate_uniform_a, "b": estimate_uniform_b},
    },
    {
        "sample_key": "uniform_1000",
        "label": "U(2,7), n=1000",
        "true_parameters": {"a": 2.0, "b": 7.0},
        "estimators": {"a": estimate_uniform_a, "b": estimate_uniform_b},
    },
    {
        "sample_key": "bernoulli_100",
        "label": "Bernoulli(0.27), n=100",
        "true_parameters": {"p": 0.27},
        "estimators": {"p": estimate_bernoulli_p},
    },
    {
        "sample_key": "bernoulli_1000",
        "label": "Bernoulli(0.27), n=1000",
        "true_parameters": {"p": 0.27},
        "estimators": {"p": estimate_bernoulli_p},
    },
    {
        "sample_key": "binom_100",
        "label": "Bin(12,0.35), n=100",
        "true_parameters": {"m": 12.0, "p": 0.35},
        "estimators": {"m": estimate_binom_m, "p": estimate_binom_p},
    },
    {
        "sample_key": "binom_1000",
        "label": "Bin(12,0.35), n=1000",
        "true_parameters": {"m": 12.0, "p": 0.35},
        "estimators": {"m": estimate_binom_m, "p": estimate_binom_p},
    },
    {
        "sample_key": "norm_100",
        "label": "N(4.5,1.8^2), n=100",
        "true_parameters": {"mu": 4.5, "sigma": 1.8},
        "estimators": {"mu": estimate_normal_mu, "sigma": estimate_normal_sigma},
    },
    {
        "sample_key": "norm_1000",
        "label": "N(4.5,1.8^2), n=1000",
        "true_parameters": {"mu": 4.5, "sigma": 1.8},
        "estimators": {"mu": estimate_normal_mu, "sigma": estimate_normal_sigma},
    },
]


In [2]:
def format_interval(interval):
    return f"[{interval[0]:.4f}; {interval[1]:.4f}]"


summary_rows = []
detail_rows = []

for experiment in EXPERIMENTS:
    sample = samples[experiment["sample_key"]]
    estimate_chunks = []
    custom_chunks = []
    scipy_chunks = []
    max_difference = 0.0

    for offset, (parameter_name, estimator) in enumerate(experiment["estimators"].items()):
        custom_result = bootstrap_percentile(
            sample,
            estimator,
            confidence_level=CONFIDENCE_LEVEL,
            n_resamples=N_RESAMPLES,
            random_state=BOOTSTRAP_RANDOM_STATE + offset,
        )
        scipy_interval = scipy_percentile_bootstrap(
            sample,
            estimator,
            confidence_level=CONFIDENCE_LEVEL,
            n_resamples=N_RESAMPLES,
            random_state=BOOTSTRAP_RANDOM_STATE + offset,
        )

        custom_interval = custom_result["confidence_interval"]
        difference = max(
            abs(custom_interval[0] - scipy_interval[0]),
            abs(custom_interval[1] - scipy_interval[1]),
        )
        max_difference = max(max_difference, difference)

        estimate_chunks.append(f"{parameter_name}={custom_result['estimate']:.4f}")
        custom_chunks.append(f"{parameter_name}: {format_interval(custom_interval)}")
        scipy_chunks.append(f"{parameter_name}: {format_interval(scipy_interval)}")

        detail_rows.append(
            {
                "Выборка": experiment["label"],
                "Параметр": parameter_name,
                "Истинное значение": experiment["true_parameters"][parameter_name],
                "Точечная оценка": custom_result["estimate"],
                "Свой bootstrap": format_interval(custom_interval),
                "SciPy bootstrap": format_interval(scipy_interval),
                "Макс. |разность|": difference,
            }
        )

    summary_rows.append(
        {
            "Выборка": experiment["label"],
            "Истинные параметры": ", ".join(
                f"{name}={value:g}" for name, value in experiment["true_parameters"].items()
            ),
            "Точечные оценки": "; ".join(estimate_chunks),
            "Свой bootstrap (95%)": "; ".join(custom_chunks),
            "SciPy bootstrap (95%)": "; ".join(scipy_chunks),
            "Макс. |разность|": max_difference,
        }
    )

summary_df = pd.DataFrame(summary_rows)
detail_df = pd.DataFrame(detail_rows)
detail_display_df = detail_df.copy()

for column in ["Истинное значение", "Точечная оценка", "Макс. |разность|"]:
    detail_display_df[column] = detail_display_df[column].astype(float).round(4)

summary_df["Макс. |разность|"] = summary_df["Макс. |разность|"].astype(float).round(4)

display(Markdown("### Сводная таблица в формате ЛР №2"))
display(summary_df)

display(Markdown("### Детальная таблица по каждому параметру"))
display(detail_display_df)

largest_gap_row = detail_display_df.loc[detail_display_df["Макс. |разность|"].idxmax()]
display(
    Markdown(
        "**Краткий вывод.** При переходе от $n=100$ к $n=1000$ доверительные интервалы становятся уже. "
        "Наша реализация bootstrap и `scipy.stats.bootstrap(..., method='percentile')` дают близкие результаты; "
        f"наибольшее расхождение в этом эксперименте наблюдается для выборки `{largest_gap_row['Выборка']}` "
        f"и параметра `{largest_gap_row['Параметр']}` и составляет примерно {largest_gap_row['Макс. |разность|']:.4f}."
    )
)

summary_df


### Сводная таблица в формате ЛР №2

,Выборка,Истинные параметры,Точечные оценки,Свой bootstrap (95%),SciPy bootstrap (95%),Макс. |разность|
0,"U(2,7), n=100","a=2, b=7",a=1.9756; b=6.9892,a: [1.6438; 2.3668]; b: [6.6102; 7.3090],a: [1.6405; 2.3750]; b: [6.5969; 7.3154],0.0132
1,"U(2,7), n=1000","a=2, b=7",a=1.9499; b=6.9520,a: [1.8416; 2.0628]; b: [6.8350; 7.0641],a: [1.8428; 2.0636]; b: [6.8340; 7.0625],0.0016
2,"Bernoulli(0.27), n=100",p=0.27,p=0.2500,p: [0.1700; 0.3400],p: [0.1700; 0.3400],0.0000
3,"Bernoulli(0.27), n=1000",p=0.27,p=0.2500,p: [0.2240; 0.2770],p: [0.2230; 0.2770],0.0010
4,"Bin(12,0.35), n=100","m=12, p=0.35",m=10.3509; p=0.4048,m: [7.9789; 15.7990]; p: [0.2656; 0.5350],m: [8.0000; 15.7700]; p: [0.2633; 0.5384],0.0290
5,"Bin(12,0.35), n=1000","m=12, p=0.35",m=12.2600; p=0.3372,m: [10.5533; 14.7624]; p: [0.2802; 0.3932],m: [10.4875; 14.7364]; p: [0.2807; 0.3914],0.0658
6,"N(4.5,1.8^2), n=100","mu=4.5, sigma=1.8",mu=4.5379; sigma=1.7762,mu: [4.2003; 4.8850]; sigma: [1.5242; 1.9999],mu: [4.1812; 4.8863]; sigma: [1.5298; 1.9955],0.0191
7,"N(4.5,1.8^2), n=1000","mu=4.5, sigma=1.8",mu=4.4620; sigma=1.7816,mu: [4.3488; 4.5747]; sigma: [1.7036; 1.8589],mu: [4.3518; 4.5727]; sigma: [1.7020; 1.8565],0.0029


### Детальная таблица по каждому параметру

,Выборка,Параметр,Истинное значение,Точечная оценка,Свой bootstrap,SciPy bootstrap,Макс. |разность|
0,"U(2,7), n=100",a,2.00,1.9756,[1.6438; 2.3668],[1.6405; 2.3750],0.0082
1,"U(2,7), n=100",b,7.00,6.9892,[6.6102; 7.3090],[6.5969; 7.3154],0.0132
2,"U(2,7), n=1000",a,2.00,1.9499,[1.8416; 2.0628],[1.8428; 2.0636],0.0012
3,"U(2,7), n=1000",b,7.00,6.9520,[6.8350; 7.0641],[6.8340; 7.0625],0.0016
4,"Bernoulli(0.27), n=100",p,0.27,0.2500,[0.1700; 0.3400],[0.1700; 0.3400],0.0000
5,"Bernoulli(0.27), n=1000",p,0.27,0.2500,[0.2240; 0.2770],[0.2230; 0.2770],0.0010
6,"Bin(12,0.35), n=100",m,12.00,10.3509,[7.9789; 15.7990],[8.0000; 15.7700],0.0290
7,"Bin(12,0.35), n=100",p,0.35,0.4048,[0.2656; 0.5350],[0.2633; 0.5384],0.0034
8,"Bin(12,0.35), n=1000",m,12.00,12.2600,[10.5533; 14.7624],[10.4875; 14.7364],0.0658
9,"Bin(12,0.35), n=1000",p,0.35,0.3372,[0.2802; 0.3932],[0.2807; 0.3914],0.0018


**Краткий вывод.** При переходе от $n=100$ к $n=1000$ доверительные интервалы становятся уже. Наша реализация bootstrap и `scipy.stats.bootstrap(..., method='percentile')` дают близкие результаты; наибольшее расхождение в этом эксперименте наблюдается для выборки `Bin(12,0.35), n=1000` и параметра `m` и составляет примерно 0.0658.

,Выборка,Истинные параметры,Точечные оценки,Свой bootstrap (95%),SciPy bootstrap (95%),Макс. |разность|
0,"U(2,7), n=100","a=2, b=7",a=1.9756; b=6.9892,a: [1.6438; 2.3668]; b: [6.6102; 7.3090],a: [1.6405; 2.3750]; b: [6.5969; 7.3154],0.0132
1,"U(2,7), n=1000","a=2, b=7",a=1.9499; b=6.9520,a: [1.8416; 2.0628]; b: [6.8350; 7.0641],a: [1.8428; 2.0636]; b: [6.8340; 7.0625],0.0016
2,"Bernoulli(0.27), n=100",p=0.27,p=0.2500,p: [0.1700; 0.3400],p: [0.1700; 0.3400],0.0000
3,"Bernoulli(0.27), n=1000",p=0.27,p=0.2500,p: [0.2240; 0.2770],p: [0.2230; 0.2770],0.0010
4,"Bin(12,0.35), n=100","m=12, p=0.35",m=10.3509; p=0.4048,m: [7.9789; 15.7990]; p: [0.2656; 0.5350],m: [8.0000; 15.7700]; p: [0.2633; 0.5384],0.0290
5,"Bin(12,0.35), n=1000","m=12, p=0.35",m=12.2600; p=0.3372,m: [10.5533; 14.7624]; p: [0.2802; 0.3932],m: [10.4875; 14.7364]; p: [0.2807; 0.3914],0.0658
6,"N(4.5,1.8^2), n=100","mu=4.5, sigma=1.8",mu=4.5379; sigma=1.7762,mu: [4.2003; 4.8850]; sigma: [1.5242; 1.9999],mu: [4.1812; 4.8863]; sigma: [1.5298; 1.9955],0.0191
7,"N(4.5,1.8^2), n=1000","mu=4.5, sigma=1.8",mu=4.4620; sigma=1.7816,mu: [4.3488; 4.5747]; sigma: [1.7036; 1.8589],mu: [4.3518; 4.5727]; sigma: [1.7020; 1.8565],0.0029
